In [6]:
import os
import math
import warnings
from dataclasses import dataclass
from typing import Tuple, Dict, List, Any, Optional, Required

import torch
from torch.quasirandom import SobolEngine
from torch import Tensor, float32

from botorch.acquisition import \
    (
        qExpectedImprovement, 
        qLogExpectedImprovement,
    )
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.fit import fit_gpytorch_mll
from botorch.generation import MaxPosteriorSampling
from botorch.models import SingleTaskGP
from botorch.optim import optimize_acqf
from botorch.test_functions import Ackley
from botorch.utils.transforms import unnormalize

import gpytorch
from gpytorch.constraints import Interval
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood

warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

device: torch.device = torch.device("mps" if torch.mps.is_available() else "cpu")
dtype: torch.dtype = float32
SMOKE_TEST: int = os.environ.get("SMOKE_TEST", 20)

In [7]:
ackley: Tensor = Ackley(dim=20, negate=True).to(dtype=dtype, device=device)
ackley.bounds[0, :].fill_(-5)
ackley.bounds[1, :].fill_(10)
dim: int = ackley.dim
lb, ub = ackley.bounds

batch_size: int = 4
n_init: int = 2 * dim
max_cholesky_size: float = float("inf")

def eval_objective(
        x: Tensor[float32]
        ):
    return ackley(unnormalize(x, ackley.bounds))

In [8]:
@dataclass
class TurboState:
    dim: int
    batch_size: int
    
    length: float = 0.8
    length_min: float = 0.5 ** 7
    length_max: float = 1.6

    failure_counter: int = 0
    failure_tolerance: int = float("nan")
    success_counter: int = 0
    success_tolerance: int = 10

    best_value: float = -float("inf")
    restart_triggered: bool = False

    def __post_init__(self):
        self.failure_tolerance = math.ceil(
            max([4.0 / self.batch_size, float(self.dim) / self.batch_size])
        )
    
def update_state(
        state: TurboState,
        Y_next: Tensor[float32],
):
    if max(Y_next) > state.best_value + 1e-3 * math.fabs(state.best_value):
        state.success_counter += 1
        state.failure_counter = 0
    else:
        state.success_counter = 0
        state.failure_counter += 1
    
    if state.success_counter == state.success_tolerance: # Expand trust region
        state.length = min(2.0 * state.length, state.length_max)
        state.success_counter
    else:
        state.length /= 2.0
        state.failure_counter = 0
    
    state.best_value = max(state.best_value, max(Y_next).item())
    if state.length < state.length_min:
        state.restart_triggered = True
    return state

state = TurboState(dim=dim, batch_size=batch_size)
print(state)


TurboState(dim=20, batch_size=4, length=0.8, length_min=0.0078125, length_max=1.6, failure_counter=0, failure_tolerance=5, success_counter=0, success_tolerance=10, best_value=-inf, restart_triggered=False)


In [9]:
def get_initial_points(
        dim: int, 
        n_pts: int, 
        seed: float = 0
        ):
    sobol: SobolEngine = SobolEngine(
        dimension=dim, 
        scramble=True, 
        seed=seed)
    X_init: Tensor = sobol.draw(
        n=n_pts
        ).to(
            dtype=dtype, 
            device=device
            )
    return X_init

def generate_batch(
        state: TurboState,
        model,
        X: Tensor,
        Y: Tensor,
        batch_size: int,
        n_candidates: int = None,
        num_restarts: int = 10,
        raw_samples: int = 512,
        acqf: str = "ts",
):
    assert acqf in ("ts", "ei")
    assert X.min() >= 0.0 and X.max() <= 1.0 and torch.all(torch.isfinite(Y))

    if n_candidates is None:
        n_candidates: int = min(500, max(2000, 200 * X.shape[-1]))
        
    x_center: Tensor = X[Y.argmax(), :].clone()
    
    weights: Tensor = model.covar_module.base_kernel.lengthscale.squeeze().detach()
    weights: Tensor = weights / weights.mean()
    weights: Tensor = weights / torch.prod(weights.pow(1.0 / len(weights)))
    
    tr_lowerbound: Tensor = torch.clamp(x_center - weights * state.length / 2.0, 0.0, 1.0)
    tr_upperbound: Tensor = torch.clamp(x_center + weights * state.length / 2.0, 0.0, 1.0)

    match acqf:
        case "ts":
            dim: int = X.shape[-1]
            sobol: SobolEngine = SobolEngine(dim, scramble=True)
            perturbation: Tensor = sobol.draw(n_candidates).to(dtype=dtype, device=device)
            perturbation: Tensor = tr_lowerbound + (tr_upperbound - tr_lowerbound) * perturbation

            prob_perturbation: float = min(20.0 / dim, 1.0)
            mask: Tensor = torch.randn(n_candidates, dim, dtype=dtype, device=device) <= prob_perturbation
            ind: Tensor = torch.where(mask.sum(dim=1) == 0)[0]
            mask[ind, torch.randint(0, dim - 1, size=(len(ind),), device=device)] = 1

            X_candidate: Tensor = x_center.expand(n_candidates, dim).clone()
            X_candidate[mask] = perturbation[mask]

            thompson_sampling: Tensor = MaxPosteriorSampling(model=model, replacement=False)
            with torch.no_grad():
                X_next: Tensor = thompson_sampling(X_candidate, num_samples=batch_size)
        case "ei":
            ei: qExpectedImprovement = qExpectedImprovement(model, train_Y.max())
            X_next, acq_value = optimize_acqf(
                ei,
                bounds=torch.stack([tr_lowerbound, tr_upperbound]),
                q=batch_size,
                num_restarts=num_restarts,
                raw_samples=raw_samples,
            )
        
        
    return X_next


In [ ]:
X_turbo: Tensor = get_initial_points(dim, n_init) # Sample initial guesses using sobol, begining exploration of search space
Y_turbo: Tensor = torch.tensor(
    [eval_objective(x) for x in X_turbo], dtype=dtype, device=device
    ).unsqueeze(-1)

state: TurboState = TurboState(dim, batch_size=batch_size, best_value=max(Y_turbo).item())

NUM_RESTARTS: int = 10 if not SMOKE_TEST else 2
RAW_SAMPLES: int = 512 if not SMOKE_TEST else 4
N_CANDIDATES: int = min(5000, max(2000, 200 * dim) if not SMOKE_TEST else 4)

torch.manual_seed(0)

while not state.restart_triggered:

    train_Y: Tensor = (Y_turbo - Y_turbo.mean()) / Y_turbo.std()
    likelihood: Tensor = GaussianLikelihood(noise_constraint=Interval(1e-8, 1e-3))
    covar_module: ScaleKernel = ScaleKernel(
        MaternKernel(
            nu=2.5, ard_num_dims=dim, lengthscale_constraint=Interval(0.005, 4.0)
        )
    )
    model: SingleTaskGP = SingleTaskGP(
        X_turbo, train_Y, covar_module=covar_module, likelihood=likelihood
    )

    # Initialize marginal log likelihood based on cached x, and y priors
    mll: ExactMarginalLogLikelihood = ExactMarginalLogLikelihood(likelihood=model.likelihood, model=model)

    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        # Fit the surrogate via the marginal log likelihood, searching for hyperparameters
        fit_gpytorch_mll(mll=mll)
        
        # query the to-predict model based on the new tuned model and acquisition function
        X_next = generate_batch(
            state=state,
            model=model,
            X=X_turbo,
            Y=train_Y,
            batch_size=batch_size,
            n_candidates=N_CANDIDATES,
            num_restarts=NUM_RESTARTS,
            raw_samples=RAW_SAMPLES,
            acqf="ts"
        )

        Y_next: Tensor = torch.tensor(
            [eval_objective(x) for x in X_next], dtype=dtype, device=device
        ).unsqueeze(-1)

        state = update_state(state=state, Y_next=Y_next)

        X_turbo = torch.cat((X_turbo, X_next), dim=0)
        Y_turbo = torch.cat((Y_turbo, Y_next), dim=0)

        print(
            f"({len(X_turbo)}) Best value: {state.best_value:.2f}, TR length: {state.length:.2f}"
        )

/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_61180/1745562301.py:23: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: SingleTaskGP = SingleTaskGP(


(44) Best value: -1.24e+01, TR length: 4.00e-01


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_61180/1745562301.py:23: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: SingleTaskGP = SingleTaskGP(


(48) Best value: -1.03e+01, TR length: 2.00e-01


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_61180/1745562301.py:23: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: SingleTaskGP = SingleTaskGP(


...

(60) Best value: -9.78e+00, TR length: 2.50e-02


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_61180/1745562301.py:23: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: SingleTaskGP = SingleTaskGP(


(64) Best value: -9.78e+00, TR length: 1.25e-02


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_61180/1745562301.py:23: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: SingleTaskGP = SingleTaskGP(
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch/fit.py:241: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch/fit.py:241: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_

ModelFittingError: All attempts to fit the model have failed.